## ライブラリの読み込み

In [ ]:
import random
from beamngpy import BeamNGpy, Scenario, Vehicle, set_up_simple_logging
import os
from pathlib import Path
import json

## Trajectoryの読み込み

In [ ]:
# JSONファイルがあるディレクトリを指定
base_path = Path("/home/apollo-22/offroad/beamng_data_collector/route_planning/data/scripts")

# ディレクトリ内のJSONファイル一覧を取得
json_files = sorted([f for f in base_path.glob("*.json")])

if not json_files:
  print("指定ディレクトリにJSONファイルが見つかりません。")
else:
  print("利用可能なJSONファイル:")
  for idx, f in enumerate(json_files):
    print(f" [{idx}] {f.name}")

  # ユーザーに選択させる
  while True:
    try:
      selection = int(input("使用するファイルの番号を入力してください: "))
      if 0 <= selection < len(json_files):
        selected_file = json_files[selection]
        print(f"選択されたファイル: {selected_file}")
        break
      else:
        print("番号が範囲外です。")
    except ValueError:
      print("数字を入力してください。")


In [ ]:
with selected_file.open("r") as f:
  data = json.load(f)

# メタデータ
vehicle_model = data["metadata"]["vehicle_model"]
map_name = data["metadata"]["map"]
spawn_pos = tuple(data["metadata"]["spawn_pos"])
spawn_rot = tuple(data["metadata"]["spawn_rot"])

# script
scripts = data["script"]

# 確認
print("vehicle_model:", vehicle_model)
print("map_name:", map_name)
print("spawn_pos:", spawn_pos)
print("spawn_rot:", spawn_rot)
print("scriptの点数:", len(scripts))

## BeamNGの起動

In [ ]:
random.seed(1703)
set_up_simple_logging()

beamng = BeamNGpy('localhost', 25252)
bng = beamng.open(launch=False)

In [ ]:
vehicle = Vehicle('ego_vehicle', model=vehicle_model, licence='ego_vehicle')

scenario = Scenario(map_name, 'LiDAR_demo', description='Spanning the map with a LiDAR sensor')

# Add the vehicle to the scenario with the specified initial position and orientation  
scenario.add_vehicle(vehicle, cling=True,
  pos=spawn_pos,  # Initial position (x, y, z)  
  rot_quat=spawn_rot  # Initial orientation as a quaternion (x, y, z, w)  
)

scenario.make(bng)
bng.settings.set_deterministic(60)
bng.load_scenario(scenario)
bng.ui.hide_hud()
bng.scenario.start()

## 自動運転

In [ ]:
vehicle.ai.set_script(scripts, cling=True)